In [1]:
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import copy
import numpy as np
import js
import scipy.stats as stats
import seaborn as sns
import itertools

from _helper_functions import * 

In [2]:
rep=1

In [3]:
with open('data/OLSEnSeq2Name.pydict.pickle','rb') as f: OLSEnSeq2Name=pickle.load(f)
dprint(OLSEnSeq2Name)

CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6


In [4]:
OLSEnName2Seq={v:k for k,v in OLSEnSeq2Name.items()}
dprint(OLSEnName2Seq)

S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6 CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA


In [5]:
def can_be_converted_to_int(val):
    try: 
        int(val)
        return True
    except ValueError:
        return False

can_be_converted_to_int('1')

True

In [6]:
TfbsOri2Name={
    'GF':'G',
    'GR':'g',
    'EF':'E',
    'ER':'e'
}

In [7]:
S1,S2,S3,S4,S5 = 'CATCTGAAGCTC','TA','TTTCGAA','TGCTC',''
G1R,G2R,G3F       = 'GTTATCTC','AATATCT','AAGATAGG'
G1F,G2F,G3R       = [revcomp(i) for i in [G1R,G2R,G3F]]
E1F,E2F           = 'ACGGAAGT','AAGGAAAT'
E1R,E2R           = [revcomp(i) for i in [E1F,E2F]]

SiteSeq2Name={
    S1 :'S1',
    G1R:'G1R',
    S2 :'S2',
    E1F:'E1F',
    S3 :'S3',
    E2F:'E2F',
    S4 :'S4',
    G2R:'G2R',
    S5 :'S5',
    G3F:'G3F',
    G1F:'G1F',
    E1R:'E1R',
    E2R:'E2R',
    G2F:'G2F',
    G3R:'G3R',
    'A':'S6'
}

SiteName2Seq={name:seq for seq,name in SiteSeq2Name.items()}

In [8]:

def parse_ord_ori_spc(sitelist):
    sitelist=sitelist.split('-')
    sentence=[]
    for sitei,site in enumerate(sitelist):
        
        # if spacer
        if site[0]=='S': 
            spacerLen=len(SiteName2Seq[site])+8
            
            # adjust for G2
            if sitei==len(sitelist)-1:     pass
            elif sitelist[sitei+1]=='G2F': spacerLen-=1
            elif sitelist[sitei-1]=='G2R': spacerLen-=1
                
            sentence.append(spacerLen)        
            
        # if tfbs
        else:
            siteWithOri=site[0]+site[2]
            siteWithOri=TfbsOri2Name[siteWithOri]
            sentence.append(siteWithOri)
            
    return sentence[1:-1] # remove flanking affs


parse_ord_ori_spc('S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6')

['g', 10, 'E', 15, 'E', 13, 'g', 7, 'G']

In [9]:

def get_all_phrases(sentence,spacing,length):
    
    # If your phrases don't include spacing
    if spacing==False:
        phrase2count={}
        for phrase in itertools.combinations(sentence, length):
            if phrase not in phrase2count:
                phrase2count[phrase]=0
            phrase2count[phrase]+=1
        return phrase2count
    
    # If your phrases include spacing
    if spacing==True:
        phrase2count={}
        sentence_withoutSpacing=[site for site in sentence if type(site)==str]
        for phrase_ilist in itertools.combinations(range(len(sentence_withoutSpacing)), length):
            
            # Get the sentence with spacing
            phrase_ilist=[2*pi for pi in phrase_ilist]
            phrase_withSpacing=[]
            for i,pi in enumerate(phrase_ilist):
                
                # if you've reached the last binding site
                if i==len(phrase_ilist)-1:
                    phrase_withSpacing.append(sentence[pi])
                
                # if there are still other spacings to calculate
                else:
                    
                    # add the first site
                    phrase_withSpacing.append(sentence[pi])
                    
                    # determine the spacing
                    pi_next=phrase_ilist[i+1]
                    
                    # add spacing
                    spacingBetweenFirstLast=sum([s for s in sentence[pi:pi_next] if type(s)==int])
                    phrase_withSpacing.append(spacingBetweenFirstLast)
                
            phrase_withSpacing=tuple(phrase_withSpacing)
            if phrase_withSpacing not in phrase2count: 
                phrase2count[phrase_withSpacing]=0
            phrase2count[phrase_withSpacing]+=1
                
        return phrase2count
        

In [10]:
sitelist='S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6'
sentence=parse_ord_ori_spc(sitelist)
print(sentence)
print()
get_all_phrases(sentence,spacing=True,length=2)

['g', 10, 'E', 15, 'E', 13, 'g', 7, 'G']



{('g', 10, 'E'): 1,
 ('g', 25, 'E'): 1,
 ('g', 38, 'g'): 1,
 ('g', 45, 'G'): 1,
 ('E', 15, 'E'): 1,
 ('E', 28, 'g'): 1,
 ('E', 35, 'G'): 1,
 ('E', 13, 'g'): 1,
 ('E', 20, 'G'): 1,
 ('g', 7, 'G'): 1}

# Load library data

In [11]:
fn='000a-rep-by-rep-activities/endf.perRepActivities.pd.df.pickle'
allDF=pd.read_pickle(fn).set_index('wt-enhancer-seq')
allDF.head()

,enhancer-id,enhancer-sites,abl-enhancer-seq,enhancer-dep-ets-gata-1,enhancer-dep-ets-gata-2,enhancer-dep-ets-gata-3,wt-pool-sumR/sumD-1-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,wt-pool-sumR/sumD-2-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,wt-pool-sumR/sumD-3-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-1-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-2-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-3-filtNa-Dnotnull-minActFilt0.15-log2-rescaled
wt-enhancer-seq,,,,,,,,,,,,
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,OLS:0,S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0,0,0,1.210796,-0.356547,1.214739,NaN,1.236959,0.979943
CATCTGAAGCTCGAGATAACTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,OLS:1,S1-G1F-S2-E1F-S3-E2F-S4-G2R-S5-G3F-S6,CATCTGAAGCTCGAAATAACTAACGCAAGTTTTCGAAAAGCAAATT...,0,0,0,-0.030701,0.624436,1.144009,NaN,-0.583672,NaN
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAGATATTAAGATAGGA,OLS:2,S1-G1R-S2-E1F-S3-E2F-S4-G2F-S5-G3F-S6,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0,0,0,1.454324,NaN,NaN,1.400798,NaN,NaN
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTCCTATCTTA,OLS:3,S1-G1R-S2-E1F-S3-E2F-S4-G2R-S5-G3R-S6,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0,0,0,NaN,NaN,NaN,-0.260262,NaN,-0.050314
CATCTGAAGCTCGTTATCTCTAACTTCCGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,OLS:4,S1-G1R-S2-E1R-S3-E2F-S4-G2R-S5-G3F-S6,CATCTGAAGCTCGTTATTTCTAACTTGCGTTTTCGAAAAGCAAATT...,0,0,1,NaN,0.082703,1.153413,-1.065569,-0.858162,-0.255821


In [12]:
len(allDF)

460800

# Generate all phrases

In [13]:
# if running for the first time 

# start dataframe
c2v={}
c2v['phrase']=[]
c2v['phrase-type']=[]
c2v['phrase-length']=[]
c2v['phrase-is-spacing']=[]

Type2Phrase2Enhancer2Count={}

for length in [2,3]:
    for spacing,parse_func,title in [(True, parse_ord_ori_spc,'parse_ord_ori_spc')]:
        
        print(length,title)
        
        if title not in Type2Phrase2Enhancer2Count:
            Type2Phrase2Enhancer2Count[title]={}
            
        orientation_bool= 'ori' in title
        
        setPhrases=set()
        for seq,sentence in OLSEnSeq2Name.items():
            sentence=parse_func(sentence)
            phrase2count=get_all_phrases(sentence,spacing,length)
            
            for phrase,count in phrase2count.items():
                
                # Check if revcomp has been detected. If so, adjusst phrase to look like rev comp.
                if syntax_revcomp(phrase,orientation_bool) in Type2Phrase2Enhancer2Count[title]:
                    phrase=syntax_revcomp(phrase,orientation_bool)
                    
                # Add phrase if not around
                if (phrase not in Type2Phrase2Enhancer2Count[title]):
                    Type2Phrase2Enhancer2Count[title][phrase]={}
                
                if seq not in Type2Phrase2Enhancer2Count[title][phrase]:
                    Type2Phrase2Enhancer2Count[title][phrase][seq]=0
                Type2Phrase2Enhancer2Count[title][phrase][seq]+=count
                
                setPhrases.add(phrase)
            
            
        for phrase in setPhrases:
            c2v['phrase'].append(phrase)
            c2v['phrase-type'].append(title)
            c2v['phrase-length'].append(length)
            c2v['phrase-is-spacing'].append(spacing)

with open('F05xi-Syntax-Features-By-Reps/Type2Phrase2Enhancer2Count.pydict.pickle','wb') as f: pickle.dump(Type2Phrase2Enhancer2Count,f,protocol=pickle.HIGHEST_PROTOCOL)

piDF=pd.DataFrame(c2v)
piDF.to_pickle(f'F05xi-Syntax-Features-By-Reps/1-all-phrases-possible.pd.df.pickle')

In [14]:
# if you already ran the above cell
with open('F05xi-Syntax-Features-By-Reps/Type2Phrase2Enhancer2Count.pydict.pickle','rb') as f: Type2Phrase2Enhancer2Count=pickle.load(f)

piDF=pd.read_pickle('F05xi-Syntax-Features-By-Reps/1-all-phrases-possible.pd.df.pickle')
piDF.head()

,phrase,phrase-type,phrase-length,phrase-is-spacing
0,"(g, 46, g)",parse_ord_ori_spc,2,True
1,"(G, 10, e)",parse_ord_ori_spc,2,True
2,"(e, 52, E)",parse_ord_ori_spc,2,True
3,"(e, 41, E)",parse_ord_ori_spc,2,True
4,"(E, 44, G)",parse_ord_ori_spc,2,True


## Determine how many ens associated are func, ? and nonfunc

In [15]:
geDF=allDF.loc[allDF[f'enhancer-dep-ets-gata-{rep}']==1,:].copy(deep=True)
print(len(geDF))
geDF.head(1)

20042


,enhancer-id,enhancer-sites,abl-enhancer-seq,enhancer-dep-ets-gata-1,enhancer-dep-ets-gata-2,enhancer-dep-ets-gata-3,wt-pool-sumR/sumD-1-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,wt-pool-sumR/sumD-2-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,wt-pool-sumR/sumD-3-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-1-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-2-filtNa-Dnotnull-minActFilt0.15-log2-rescaled,abl-pool-sumR/sumD-3-filtNa-Dnotnull-minActFilt0.15-log2-rescaled
wt-enhancer-seq,,,,,,,,,,,,
CATCTGAAGCTCGAGATAACTAACTTCCGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,OLS:8,S1-G1F-S2-E1R-S3-E2F-S4-G2R-S5-G3F-S6,CATCTGAAGCTCGAAATAACTAACTTGCGTTTTCGAAAAGCAAATT...,1,0,0,1.441856,0.478403,0.987567,-0.822986,NaN,NaN


In [16]:
active_seqs_dependent_gataets=set(geDF.index)
list(active_seqs_dependent_gataets)[:3]

['TAACGGAAGTTGCTCAGATATTTTTCGAACCTATCTTCATCTGAAGCTCATTTCCTTGTTATCTCA',
 'TTTCGAAGTTATCTCTAATTTCCTTCATCTGAAGCTCACTTCCGTAGATATTTGCTCCCTATCTTA',
 'TTTCGAAATTTCCTTTGCTCAATATCTACGGAAGTCATCTGAAGCTCAAGATAGGTAGTTATCTCA']

In [17]:
handle_activity='not_detected_skip'

handle_activity_counts = 'num_activity_added_weighted_by_phrase_counts'

# Which activity to use
activityCol=f'wt-pool-sumR/sumD-{rep}-filtNa-Dnotnull-minActFilt0.15-log2-rescaled'

confidentPos=1
confidentNeg=0

In [18]:
## Remove func variants not dependent on gata and ets

# what are all active variants?
active_seqs_all_gataets = set(allDF.loc[allDF[activityCol]>=confidentPos,:].index)

# what active variants not dependent on gata and ets
active_seqs_independent_gataets = active_seqs_all_gataets - active_seqs_dependent_gataets

# remove active variants not dependent on gata and ets
desired_seqs = set(allDF.index) - active_seqs_independent_gataets

# subset
allDF=allDF.loc[list(desired_seqs),:]
len(allDF)

421617

In [19]:
# if running for the first time 

# get real data for phrases
realDataList=[]

for phrase,title in zip(piDF['phrase'],piDF['phrase-type']):
    
    enActivityList=[]
    
    orientation_bool='ori' in title
    
    # You only need to consider the fwd because the dictionary has the enhancers mapped to both the fwd and rev motif already
    for enSeq in Type2Phrase2Enhancer2Count[title][phrase]:

        if handle_activity=='not_detected_skip':

            # skip if not detected
            if enSeq not in allDF.index: continue

            # add activity only once 
            if    handle_activity_counts=='num_activity_added_weighted_by_phrase_counts':
                for i in range(Type2Phrase2Enhancer2Count[title][phrase][enSeq]):
                    enActivityList.append(allDF.at[enSeq,activityCol])

            # add activity multiple times
            elif  handle_activity_counts=='one_activity_per_phrase':
                enActivityList.append(allDF.at[enSeq,activityCol])

            else: raise ValueError('paramter not acounted for')



        else: raise ValueError('parameter not accounted for 2')
    
    nFuncEns=sum([act>confidentPos for act in enActivityList])
    nNfuncEns=sum([act<confidentNeg for act in enActivityList])
    nNotConfident=len(enActivityList)-nFuncEns-nNfuncEns
    
    realDataList.append((nFuncEns,nNotConfident,nNfuncEns))
    
piDF['real-data']=realDataList
piDF.to_pickle(f'F05xi-Syntax-Features-By-Reps/2-rep{rep}-all-phrases-possible-withRealData-50k.pd.df.pickle')


In [20]:
# if you already ran the above cell once 
piDF=pd.read_pickle(f'F05xi-Syntax-Features-By-Reps/2-rep{rep}-all-phrases-possible-withRealData-50k.pd.df.pickle')
piDF.head()


,phrase,phrase-type,phrase-length,phrase-is-spacing,real-data
0,"(g, 46, g)",parse_ord_ori_spc,2,True,"(305, 2564, 1262)"
1,"(G, 10, e)",parse_ord_ori_spc,2,True,"(1778, 34023, 15806)"
2,"(e, 52, E)",parse_ord_ori_spc,2,True,"(106, 1362, 625)"
3,"(e, 41, E)",parse_ord_ori_spc,2,True,"(13, 491, 226)"
4,"(E, 44, G)",parse_ord_ori_spc,2,True,"(346, 6629, 3063)"


In [21]:
len(piDF)

9022

In [22]:
filt_name='ord_ori_spc_2-3mer'

filtLength=(piDF['phrase-length'].isin([2,3]))
filtType=(piDF['phrase-type'].isin(['parse_ord_ori_spc']))
piDF=piDF.loc[filtLength & filtType,:]

In [23]:
len(piDF)

9022

## Determine significance of enrichment using hypergeometric

In [24]:
phraseKeyList=[]

for idx in piDF.index:
    phraseTuple=piDF.at[idx,'phrase']
    phraseTuple='_'.join([str(i) for i in phraseTuple])
    
    phraseType=piDF.at[idx,'phrase-type']
    phraseType=phraseType.replace('parse_','')
    
    phraseKey=f'{phraseType}={phraseTuple}'
    phraseKeyList.append(phraseKey)
    
    
piDF['phrase-key']=phraseKeyList
piDF=piDF.set_index('phrase-key')
piDF.head()

,phrase,phrase-type,phrase-length,phrase-is-spacing,real-data
phrase-key,,,,,
ord_ori_spc=g_46_g,"(g, 46, g)",parse_ord_ori_spc,2,True,"(305, 2564, 1262)"
ord_ori_spc=G_10_e,"(G, 10, e)",parse_ord_ori_spc,2,True,"(1778, 34023, 15806)"
ord_ori_spc=e_52_E,"(e, 52, E)",parse_ord_ori_spc,2,True,"(106, 1362, 625)"
ord_ori_spc=e_41_E,"(e, 41, E)",parse_ord_ori_spc,2,True,"(13, 491, 226)"
ord_ori_spc=E_44_G,"(E, 44, G)",parse_ord_ori_spc,2,True,"(346, 6629, 3063)"


In [25]:
# example of test

# lets say we have 100 marbles total

# we have 10 red marbles and 90 black marbles

# we sample 30 marbles

# what is the p value of getting 5 marbles? 
# (ie, what is probability of getting 5 or more red marbles assuming H0 of 100 marbles total with 10 red?)

# survival function allows us to get P(X≥k)

# sanity checked using this calculator https://stattrek.com/online-calculator/hypergeometric

M,n,N,k=100,10,30,5
stats.hypergeom.sf(k=k, M=M, n=n, N=N)

0.03876507128330702

In [26]:
# set parameters for hypergeometric
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html

M      = len(allDF)                             # total number of objects            (ie total number enhancers)
n_func = sum(allDF[activityCol]>confidentPos)   # total number of type i objects     (ie total number of func ens)
n_nfun = sum(allDF[activityCol]<confidentNeg)   # total number of type i objects     (ie total number of nonfunc ens)

M,n_func,n_nfun


(421617, 20042, 123167)

In [27]:
# Given the min is 16, I might opt for 50 and see how that goes
N_minthreshold=50

In [28]:
# Setting up effect size table
#
# Recall that...
#    n = total number functional enhancers in lib (50,643)
#    M = total number sequences in lib (407147)
#    N = number sequences with motif
#    k = number functional enhancers with motif

#######################################################
#           Func En (F)             Non Func Seq (N)
#######################################################
#
# Motif     Fm = k                     Nm = N-k 
#
#######################################################
#
# No Motif  Fn = n-k                  Nn = M - n - Nm
#
#######################################################

# Odds Ratio = OR = (Fm/Nm) / (Fn/Nn)
# Rel. Risk  = RR = (Fm/(Fm+Nm)) / (Fn/(Fn+Nn))

In [29]:
percentFuncBackground=n_func/M
percentNfunBackground=n_nfun/M

pvalsF =[] # p value of func enrichment
enrichF=[]
orF=[]


for mi in piDF.index:
    numF,numU,numNF=piDF.at[mi,'real-data']
    
    N = numF+numU+numNF # sampled number of objects     (ie number of enhancers with motif)
    
    if N>=N_minthreshold:
        
        # Getting pvalue
        k_func = numF          # sampled number type i objects (ie number of enhancers with motif that are func)
        k_nfun = numNF         # sampled number type i objects (ie number of enhancers with motif that are non func)
        
        # Effect size as fold change
        percentFuncForeground = numF / N
        funcEnrich            = percentFuncForeground / percentFuncBackground
        
        # Calculate params for OR and RR
        Fm_f=k_func # Fm = k        
        Nm_f = N-k_func # Nm = N-k
        Fn_f = n_func - k_func # Fn = n-k
        Nn_f = M - n_func - Nm_f # Nn = M - n - Nm
        
        # Effect size as OR
        OR_f = (Fm_f/Nm_f) / (Fn_f/Nn_f) # OR = (Fm/Nm) / (Fn/Nn)
        

        pvalsF.append(stats.hypergeom.sf(k=k_func, M=M, n=n_func, N=N))
        enrichF.append(funcEnrich)
        orF.append(OR_f)
        
    else:
        pvalsF .append(np.NaN)
        enrichF.append(np.NaN)
        orF.append(np.NaN)


In [30]:
piDF['hyp-func-p']  = pvalsF 
piDF['func-enrich']  = enrichF 
piDF['func-odds-ratio']       = orF 
piDF.head()

,phrase,phrase-type,phrase-length,phrase-is-spacing,real-data,hyp-func-p,func-enrich,func-odds-ratio
phrase-key,,,,,,,,
ord_ori_spc=g_46_g,"(g, 46, g)",parse_ord_ori_spc,2,True,"(305, 2564, 1262)",4.432536e-14,1.553180,1.606508
ord_ori_spc=G_10_e,"(G, 10, e)",parse_ord_ori_spc,2,True,"(1778, 34023, 15806)",1.000000e+00,0.724770,0.687200
ord_ori_spc=e_52_E,"(e, 52, E)",parse_ord_ori_spc,2,True,"(106, 1362, 625)",2.329938e-01,1.065402,1.069258
ord_ori_spc=e_41_E,"(e, 41, E)",parse_ord_ori_spc,2,True,"(13, 491, 226)",9.999842e-01,0.374626,0.362874
ord_ori_spc=E_44_G,"(E, 44, G)",parse_ord_ori_spc,2,True,"(346, 6629, 3063)",1.000000e+00,0.725113,0.710299


In [31]:
# Remove chi squarre tests with not enough counts to do proper test
keepList= (piDF['hyp-func-p'].notnull())
statsDF=piDF.loc[keepList,:].copy(deep=True)
len(statsDF)

9022

In [32]:
# Remove tests with <10% enrichment
enrichmentDeltaRequired_func=.1
FuncEnrichment    = (statsDF['func-enrich'   ]-1 >= enrichmentDeltaRequired_func)
statsDF=statsDF.loc[FuncEnrichment]

len(statsDF),'valid tests'

(3208, 'valid tests')

In [33]:
statsDF['hyp-func-p-bonf']  = statsDF['hyp-func-p']  .apply(lambda p:min(1, p*len(statsDF)))

In [34]:
sigLevel=0.05

In [35]:
numSig=len(statsDF.loc[statsDF['hyp-func-p-bonf']<sigLevel])
len(piDF),numSig,percent(numSig/len(piDF))
# numSig

(9022, 386, '4.3%')

## Write out stats

In [36]:
# save to pickle and csv
outDF=statsDF.sort_values('func-enrich',ascending=False)
outDF['phrase-type']=outDF['phrase-type'].apply(lambda s: s.replace('parse_',''))

# all phrase lengths
fn=f'F05xi-Syntax-Features-By-Reps/4-rep{rep}-stats-all-phrases-possible-realData-50k_filtMotifs={filt_name}_minenrichFUNC={enrichmentDeltaRequired_func}_minN={N_minthreshold}'
outDF.to_pickle(fn+'.pandas.df.pickle')
outDF.to_csv(fn+'.tsv',index=False,sep='\t')

In [37]:
sigmotifs= (statsDF['hyp-func-p-bonf']<sigLevel)
sigDF=statsDF.loc[sigmotifs,:].sort_values('func-enrich',ascending=False)
sigDF.head()

,phrase,phrase-type,phrase-length,phrase-is-spacing,real-data,hyp-func-p,func-enrich,func-odds-ratio,hyp-func-p-bonf
phrase-key,,,,,,,,,
ord_ori_spc=g_21_e_24_E,"(g, 21, e, 24, E)",parse_ord_ori_spc,3,True,"(28, 50, 12)",7.994878e-17,6.544743,9.060081,2.564757e-13
ord_ori_spc=g_36_g_9_E,"(g, 36, g, 9, E)",parse_ord_ori_spc,3,True,"(68, 173, 56)",3.910492e-28,4.816477,5.966605,1.254486e-24
ord_ori_spc=g_13_e_24_E,"(g, 13, e, 24, E)",parse_ord_ori_spc,3,True,"(39, 108, 35)",4.234489e-16,4.507858,5.473252,1.358424e-12
ord_ori_spc=g_33_e_24_E,"(g, 33, e, 24, E)",parse_ord_ori_spc,3,True,"(20, 57, 18)",3.268785e-09,4.428773,5.347451,1.048626e-05
ord_ori_spc=g_15_E_22_E,"(g, 15, E, 22, E)",parse_ord_ori_spc,3,True,"(38, 108, 35)",1.970859e-15,4.416539,5.332645,6.322517e-12


In [38]:
len(sigDF)

386

In [39]:
sigDF.to_pickle(f'F05xi-Syntax-Features-By-Reps/5-rep{rep}-stats-sig-motifs-possible-realData_filtMotifs={filt_name}_minenrichFUNC={enrichmentDeltaRequired_func}_minN={N_minthreshold}.pandas.df.pickle')
